# Code mẫu — Self-supervised và CLIP

CLIP đặt ảnh và câu chữ vào chung một không gian vector, để tìm ảnh bằng chữ hoặc chọn câu mô tả cho ảnh. Các vector dưới đây đều đặt bằng tay, để ta thấy rõ cosine, temperature, và vai trò của các cặp sai trong batch. Chúng không phải embedding lấy từ CLIP thật.

## Ví dụ 1: Tìm câu mô tả hợp nhất theo cosine

Có ba vector ảnh và ba vector chữ, mỗi vector hai chiều. Mỗi ảnh và câu mô tả của nó được đặt gần cùng hướng. Để tìm kiếm, ta đưa từng vector về độ dài 1, rồi tính bảng similarity. Hàng thứ i cho biết ảnh i gần từng câu tới đâu. Cột thứ j cho biết câu j gần từng ảnh tới đâu.

In [ ]:
import numpy as np

def normalize(vectors):
    vectors = np.asarray(vectors, dtype=float)
    lengths = np.linalg.norm(vectors, axis=1, keepdims=True)
    if np.any(lengths == 0):
        raise ValueError("Vector 0 không có hướng cosine")
    return vectors / lengths

images = normalize([[1, 0], [0, 1], [-1, 0]])
texts = normalize([[0.9, 0.1], [0.1, 0.9], [-0.9, 0.1]])
similarity = images @ texts.T
print("Cosine:", similarity.round(4).tolist())
print("Mô tả tốt nhất cho từng ảnh:", similarity.argmax(1).tolist())
print("Hai ảnh đầu cho mô tả 1:", np.argsort(-similarity[:, 1], kind="stable")[:2].tolist())
assert np.isclose(similarity[0, 0], 0.9 / np.sqrt(0.82))
assert similarity.argmax(1).tolist() == [0, 1, 2]
try:
    normalize([[0, 0]])
except ValueError:
    print("Vector 0: bị từ chối")
else:
    raise AssertionError("Chưa bắt vector 0")

Cosine: [[0.9939, 0.1104, -0.9939], [0.1104, 0.9939, 0.1104], [-0.9939, -0.1104, 0.9939]]
Mô tả tốt nhất cho từng ảnh: [0, 1, 2]
Hai ảnh đầu cho mô tả 1: [1, 0]
Vector 0: bị từ chối


Cosine của cặp đầu là $0.9/\sqrt{0.82}\approx0.9939$. Lấy `argmax` theo hàng, mỗi ảnh tìm đúng câu của nó. Lấy theo cột, ví dụ cột 1, ta xếp hạng các ảnh cho câu thứ hai, và ảnh 1 đứng đầu. Nhầm hàng với cột là nhầm hướng tìm kiếm. Ba cặp khớp đúng ở đây chỉ xác nhận cách chuẩn hóa và xếp hạng, trên những vector đã dựng sẵn. Nó chưa đo chất lượng của CLIP.

## Ví dụ 2: Temperature đổi độ chắc chắn, gradient kéo đúng cặp lên

Ba cặp đúng nằm trên đường chéo của bảng similarity. Các ô còn lại là cặp sai trong cùng batch, với giả định mỗi ảnh chỉ có đúng một câu đúng. Loss của CLIP lấy cross-entropy theo cả hai chiều: từ ảnh tìm chữ, và từ chữ tìm ảnh. Ta so temperature 0.5 với 0.1 trên cùng bảng similarity, rồi kiểm dấu của gradient.

In [ ]:
import torch
import torch.nn.functional as F

torch.set_num_threads(1)
images = F.normalize(torch.tensor([[1., 0.], [0., 1.], [-1., 0.]]), dim=1)
texts = F.normalize(torch.tensor([[0.9, 0.1], [0.1, 0.9], [-0.9, 0.1]]), dim=1)
similarity = images @ texts.T
target = torch.arange(3)
for temperature in [0.5, 0.1]:
    logits = (similarity / temperature).detach().requires_grad_(True)
    loss = (F.cross_entropy(logits, target) + F.cross_entropy(logits.T, target)) / 2
    loss.backward()
    print("Temperature", temperature, "loss:", round(loss.item(), 4))
    print("P(caption đúng | ảnh 0):", round(logits.softmax(1)[0, 0].item(), 4))
    print("Gradient logit đúng âm:", bool((logits.grad.diag() < 0).all()))
    print("Gradient negative [0,1] dương:", bool(logits.grad[0, 1] > 0))
    assert (logits.grad.diag() < 0).all() and logits.grad[0, 1] > 0
assert (similarity / 0.1).argmax(1).tolist() == (similarity / 0.5).argmax(1).tolist()

Temperature 0.5 loss: 0.1972
P(caption đúng | ảnh 0): 0.8406
Gradient logit đúng âm: True
Gradient negative [0,1] dương: True
Temperature 0.1 loss: 0.0002
P(caption đúng | ảnh 0): 0.9999
Gradient logit đúng âm: True
Gradient negative [0,1] dương: True


Hạ temperature từ 0.5 xuống 0.1, xác suất ảnh 0 chọn đúng câu tăng từ 0.84 lên 0.9999, dù cosine không đổi và thứ hạng cũng không đổi. Temperature chỉ làm phân phối nhọn hơn.

Gradient trên đường chéo đều âm. Đi ngược gradient sẽ làm điểm của cặp đúng tăng lên. Gradient ở ô `[0, 1]`, một cặp sai, thì dương, nên điểm của cặp sai bị đẩy xuống. Đây mới là đạo hàm theo bảng điểm. Khi train thật, gradient này còn đi tiếp vào hai encoder ảnh và chữ.

## Ví dụ 3: Hai câu cùng đúng, nhưng loss chỉ công nhận một

Một ảnh có hai câu mô tả cùng điểm 2. Nếu nhãn chỉ ghi câu 0 là đúng, loss vẫn phạt câu 1, dù nội dung câu 1 có thể cũng tả đúng ảnh. Sau đó ta thêm một ứng viên có điểm bằng ứng viên tốt nhất, để thấy softmax phụ thuộc vào tập ứng viên.

In [ ]:
import math
import torch
import torch.nn.functional as F

torch.set_num_threads(1)
logits = torch.tensor([[2., 2.]], requires_grad=True)
loss = F.cross_entropy(logits, torch.tensor([0]))
loss.backward()
print("Loss hai caption cùng điểm:", round(loss.item(), 4))
print("Gradient:", logits.grad.tolist())
assert math.isclose(loss.item(), math.log(2), rel_tol=1e-6)
assert logits.grad.tolist() == [[-0.5, 0.5]]

old = torch.tensor([2., 0.]).softmax(0)
expanded = torch.tensor([2., 0., 2.]).softmax(0)
print("Hai ứng viên:", [round(v, 4) for v in old.tolist()])
print("Thêm ứng viên trùng:", [round(v, 4) for v in expanded.tolist()])
assert expanded[0] < old[0]

Loss hai caption cùng điểm: 0.6931
Gradient: [[-0.5, 0.5]]
Hai ứng viên: [0.8808, 0.1192]
Thêm ứng viên trùng: [0.4683, 0.0634, 0.4683]


Loss bằng $\log 2$, và gradient `+0.5` ở câu thứ hai sẽ đẩy điểm của nó xuống. Nếu câu thứ hai cũng tả đúng ảnh, model đang bị dạy sai. Ta gọi đó là false negative trong batch. Batch càng lớn, khả năng có hai mẫu gần giống nhau càng cao.

Phần sau cho thấy xác suất của ứng viên đầu tụt từ 0.88 xuống 0.47, chỉ vì có thêm một ứng viên cùng điểm. Không có bước học nào, cosine cũng không đổi. Xác suất softmax luôn là tương đối so với các ứng viên đang có, nên khi đọc nó, phải nhớ đang so với những ai.